# CLIFpy Data Quality Assessment (DQA) Demo

This notebook demonstrates the usage of all DQA functions in CLIFpy's `validator` module.

## Features:
- **Polars as default backend** (memory-efficient, lazy evaluation)
- **DuckDB as fallback** if Polars is unavailable
- **Optimized for large datasets** (300GB+)

In [ ]:
# Import required libraries
import polars as pl
import pandas as pd
from datetime import datetime

# Import DQA functions from CLIFpy
from clifpy.utils.validator import (
    _load_schema,
    check_table_exists,
    check_required_columns,
    check_column_dtypes,
    check_datetime_format,
    check_lab_reference_units,
    check_categorical_values,
    check_missingness,
    check_conditional_requirements,
    check_mcide_value_coverage,
    check_relational_integrity,
    run_conformance_checks,
    run_completeness_checks,
    run_relational_integrity_checks,
    run_full_dqa,
    HAS_POLARS
)

# Import IO functions
from clifpy.utils.io import load_data, fetch_lazy_result, close_lazy_relation
from clifpy.utils.io_polars import load_data_polars

print(f"Using Polars backend: {HAS_POLARS}")

---
## Loading Data from Files

CLIFpy provides two IO modules for loading data:
- **`io_polars.py`**: Polars-based, returns `pl.LazyFrame` (default) or `pl.DataFrame`
- **`io.py`**: DuckDB-based, returns `LazyRelation` or `pd.DataFrame`

Both support lazy evaluation for memory-efficient processing of large datasets (300GB+).

In [ ]:
# Example: Loading data with Polars (io_polars.py)
# ------------------------------------------------
# Set your data directory path
data_path = '/path/to/your/clif/data'

# Lazy loading (default) - returns pl.LazyFrame
# Ideal for large datasets (300GB+) as it defers execution
# labs_lazy = load_data_polars('labs', data_path, 'parquet', lazy=True)

# Apply filters and transformations lazily (no execution yet)
# filtered_labs = labs_lazy.filter(pl.col('lab_category') == 'sodium')

# Run DQA checks on LazyFrame directly
# result = check_required_columns(filtered_labs, labs_schema, 'labs')

# Collect results when needed
# df = filtered_labs.collect()

# Eager loading - returns pl.DataFrame
# labs_df = load_data_polars('labs', data_path, 'parquet', lazy=False)

print("Polars loading patterns (commented - update data_path to run):")

In [ ]:
# Example: Loading data with DuckDB (io.py)
# -------------------------------------------
# Set your data directory path
data_path = '/path/to/your/clif/data'

# Lazy loading - returns LazyRelation (DuckDB relation wrapper)
# labs_lazy = load_data('labs', data_path, 'parquet', lazy=True)

# Apply filters using DuckDB SQL-like syntax
# filtered_labs = labs_lazy.filter("lab_category = 'sodium'")

# Fetch as pandas DataFrame when ready
# labs_pandas = fetch_lazy_result(filtered_labs, site_tz='America/New_York')

# Or fetch and close manually
# labs_pandas = filtered_labs.fetchdf()
# close_lazy_relation(filtered_labs)

# Run DQA checks on the resulting pandas DataFrame
# result = check_required_columns(labs_pandas, labs_schema, 'labs')

# Eager loading - returns pandas DataFrame directly
# labs_pandas = load_data('labs', data_path, 'parquet', lazy=False)

print("DuckDB loading patterns (commented - update data_path to run):")

## 1. Create Sample Data

We'll create sample labs data that mimics the CLIF format.

In [ ]:
# Create sample labs data using Polars
labs_df = pl.DataFrame({
    'hospitalization_id': ['H001', 'H001', 'H002', 'H002', 'H003'],
    'lab_order_dttm': [
        datetime(2024, 1, 1, 8, 0),
        datetime(2024, 1, 1, 12, 0),
        datetime(2024, 1, 2, 9, 0),
        datetime(2024, 1, 2, 15, 0),
        datetime(2024, 1, 3, 10, 0)
    ],
    'lab_collect_dttm': [
        datetime(2024, 1, 1, 8, 30),
        datetime(2024, 1, 1, 12, 30),
        datetime(2024, 1, 2, 9, 30),
        datetime(2024, 1, 2, 15, 30),
        datetime(2024, 1, 3, 10, 30)
    ],
    'lab_result_dttm': [
        datetime(2024, 1, 1, 9, 0),
        datetime(2024, 1, 1, 13, 0),
        datetime(2024, 1, 2, 10, 0),
        datetime(2024, 1, 2, 16, 0),
        datetime(2024, 1, 3, 11, 0)
    ],
    'lab_order_category': ['bmp', 'bmp', 'cbc', 'bmp', 'lft'],
    'lab_category': ['sodium', 'potassium', 'hemoglobin', 'creatinine', 'alt'],
    'lab_value': ['140', '4.2', '12.5', '1.1', '35'],
    'lab_value_numeric': [140.0, 4.2, 12.5, 1.1, 35.0],
    'reference_unit': ['mmol/L', 'mmol/L', 'g/dL', 'mg/dL', 'U/L']
})

print("Sample Labs Data:")
labs_df

## 2. Load Schema

Load the YAML schema for the labs table.

In [ ]:
# Load the labs schema
labs_schema = _load_schema('labs')

print(f"Table: {labs_schema['table_name']}")
print(f"Required columns: {labs_schema['required_columns']}")
print(f"Category columns: {labs_schema['category_columns']}")

---
## CONFORMANCE CHECKS

### A.1. Check Table Exists

In [ ]:
# Check if table file exists (this checks for file presence, not DataFrame validation)
result = check_table_exists('/path/to/data', 'clif_labs', 'parquet')

print(f"Check: {result.check_type}")
print(f"Passed: {result.passed}")
print(f"Errors: {result.errors}")
print(f"Info: {result.info}")

### A.2. Check Required Columns

In [ ]:
# Check if all required columns are present
result = check_required_columns(labs_df, labs_schema, 'labs')

print(f"Check: {result.check_type}")
print(f"Passed: {result.passed}")
print(f"Metrics: {result.metrics}")
print(f"Info: {result.info}")

### B.1. Check Column Data Types

In [ ]:
# Check if columns have correct data types
result = check_column_dtypes(labs_df, labs_schema, 'labs')

print(f"Check: {result.check_type}")
print(f"Passed: {result.passed}")
print(f"Metrics: {result.metrics}")
if result.warnings:
    print(f"Warnings: {result.warnings}")
if result.errors:
    print(f"Errors: {result.errors}")

### B.2. Check Datetime Format

In [ ]:
# Check datetime columns format
result = check_datetime_format(labs_df, labs_schema, 'labs', expected_tz='UTC')

print(f"Check: {result.check_type}")
print(f"Passed: {result.passed}")
print(f"Metrics: {result.metrics}")
print(f"Info: {result.info}")

### B.3. Check Lab Reference Units

In [ ]:
# Check if lab reference units match schema definitions
result = check_lab_reference_units(labs_df, labs_schema, 'labs')

print(f"Check: {result.check_type}")
print(f"Passed: {result.passed}")
print(f"Metrics: {result.metrics}")
if result.warnings:
    print(f"Warnings: {result.warnings}")
print(f"Info: {result.info}")

### B.4. Check Categorical Values

In [ ]:
# Check if categorical values match mCIDE permissible values
result = check_categorical_values(labs_df, labs_schema, 'labs')

print(f"Check: {result.check_type}")
print(f"Passed: {result.passed}")
print(f"Metrics: {result.metrics}")
if result.errors:
    print(f"Errors: {result.errors}")
print(f"Info: {result.info}")

---
## COMPLETENESS CHECKS

### A.1. Check Missingness

In [ ]:
# Create data with some missing values for demo
labs_with_nulls = pl.DataFrame({
    'hospitalization_id': ['H001', 'H002', None, 'H004', 'H005'],
    'lab_order_dttm': [
        datetime(2024, 1, 1, 8, 0),
        None,
        datetime(2024, 1, 2, 9, 0),
        datetime(2024, 1, 2, 15, 0),
        datetime(2024, 1, 3, 10, 0)
    ],
    'lab_collect_dttm': [
        datetime(2024, 1, 1, 8, 30),
        datetime(2024, 1, 1, 12, 30),
        datetime(2024, 1, 2, 9, 30),
        datetime(2024, 1, 2, 15, 30),
        datetime(2024, 1, 3, 10, 30)
    ],
    'lab_result_dttm': [
        datetime(2024, 1, 1, 9, 0),
        datetime(2024, 1, 1, 13, 0),
        datetime(2024, 1, 2, 10, 0),
        datetime(2024, 1, 2, 16, 0),
        datetime(2024, 1, 3, 11, 0)
    ],
    'lab_order_category': ['bmp', 'bmp', 'cbc', 'bmp', 'lft'],
    'lab_category': ['sodium', 'potassium', 'hemoglobin', 'creatinine', 'alt'],
    'lab_value': ['140', '4.2', '12.5', '1.1', '35'],
    'reference_unit': ['mmol/L', 'mmol/L', 'g/dL', 'mg/dL', 'U/L']
})

# Check missingness with custom thresholds
result = check_missingness(
    labs_with_nulls, 
    labs_schema, 
    'labs',
    error_threshold=50.0,   # Error if > 50% missing
    warning_threshold=10.0  # Warning if > 10% missing
)

print(f"Check: {result.check_type}")
print(f"Passed: {result.passed}")
print(f"Total rows: {result.metrics.get('total_rows')}")
print(f"\nMissingness stats:")
for stat in result.metrics.get('missingness_stats', [])[:5]:
    print(f"  {stat['column']}: {stat['percent_missing']}% missing")
if result.warnings:
    print(f"\nWarnings: {result.warnings}")

### A.2. Check Conditional Requirements

In [ ]:
# Create sample respiratory support data
resp_df = pl.DataFrame({
    'hospitalization_id': ['H001', 'H001', 'H002', 'H002'],
    'device_category': ['IMV', 'NIPPV', 'nasal_cannula', 'IMV'],
    'mode_category': ['AC/VC', None, None, 'SIMV'],  # IMV/NIPPV should have mode
    'fio2_set': [0.4, None, 0.21, 0.5],
    'peep_set': [5.0, None, None, 8.0]
})

# Check conditional requirements for respiratory support
result = check_conditional_requirements(resp_df, 'respiratory_support')

print(f"Check: {result.check_type}")
print(f"Passed: {result.passed}")
if result.warnings:
    print(f"\nWarnings:")
    for w in result.warnings:
        print(f"  - {w['message']}")
        print(f"    Details: {w['details']}")
print(f"\nInfo: {result.info}")

### B. Check mCIDE Value Coverage

In [ ]:
# Check if all mCIDE standardized values are present in the data
result = check_mcide_value_coverage(labs_df, labs_schema, 'labs')

print(f"Check: {result.check_type}")
print(f"Passed: {result.passed}")
print(f"Metrics: {result.metrics.get('category_columns_checked')} category columns checked")

# Show coverage by column
coverage = result.metrics.get('coverage_by_column', {})
for col, details in coverage.items():
    print(f"\n{col}:")
    print(f"  Found {details['found_values']}/{details['expected_values']} values ({details['coverage_percent']}%)")
    if details['missing_values']:
        print(f"  Missing: {details['missing_values'][:5]}..." if len(details['missing_values']) > 5 else f"  Missing: {details['missing_values']}")

### C. Check Relational Integrity (Bidirectional)

In [ ]:
# Create reference hospitalization table
hospitalization_df = pl.DataFrame({
    'hospitalization_id': ['H001', 'H002'],  # H003 is missing!
    'patient_id': ['P001', 'P002'],
    'admission_dttm': [datetime(2024, 1, 1), datetime(2024, 1, 2)]
})

# Check relational integrity (bidirectional)
result = check_relational_integrity(
    target_df=labs_df,
    reference_df=hospitalization_df,
    target_table='labs',
    reference_table='hospitalization',
    key_column='hospitalization_id'
)

print(f"Check: {result.check_type}")
print(f"Table: {result.table_name}")
print(f"Passed: {result.passed}")
print(f"\nForward (hospitalization → labs):")
print(f"  Coverage: {result.metrics['forward_coverage_percent']}%")
print(f"  Orphan IDs: {result.metrics['forward_orphan_ids']}")
print(f"  Unique ref IDs: {result.metrics['forward_reference_unique_ids']}")
print(f"\nReverse (labs → hospitalization):")
print(f"  Coverage: {result.metrics['reverse_coverage_percent']}%")
print(f"  Orphan IDs: {result.metrics['reverse_orphan_ids']}")
print(f"  Unique target IDs: {result.metrics['reverse_target_unique_ids']}")

if result.warnings:
    print(f"\nWarnings:")
    for w in result.warnings:
        print(f"  - {w['message']}")

In [ ]:
# Auto-detected relational integrity — just pass your table instances
# -------------------------------------------------------------------
# If using BaseTable instances loaded via load_data / load_data_polars:
#   results = run_relational_integrity_checks([labs, hosp, vitals, patient, ...])
#
# Demo with sample DataFrames using a lightweight wrapper:

class TableWrapper:
    """Minimal wrapper so run_relational_integrity_checks can read .table_name and .df."""
    def __init__(self, name, df):
        self.table_name = name
        self.df = df

tables = [
    TableWrapper('labs', labs_df),
    TableWrapper('hospitalization', hospitalization_df),
]

results = run_relational_integrity_checks(tables)

for tbl_name, checks in results.items():
    for fk_col, res in checks.items():
        print(f"{tbl_name}.{fk_col}: "
              f"fwd={res.metrics['forward_coverage_percent']}%, "
              f"rev={res.metrics['reverse_coverage_percent']}%")

---
## ORCHESTRATION FUNCTIONS

### Run All Conformance Checks

In [ ]:
# Run all conformance checks at once
conformance_results = run_conformance_checks(labs_df, labs_schema, 'labs')

print("=" * 50)
print("CONFORMANCE CHECK RESULTS")
print("=" * 50)

for check_name, result in conformance_results.items():
    status = "PASS" if result.passed else "FAIL"
    print(f"\n{check_name}: {status}")
    if result.errors:
        print(f"  Errors: {len(result.errors)}")
    if result.warnings:
        print(f"  Warnings: {len(result.warnings)}")

### Run All Completeness Checks

In [ ]:
# Run all completeness checks at once
completeness_results = run_completeness_checks(
    labs_with_nulls, 
    labs_schema, 
    'labs',
    error_threshold=50.0,
    warning_threshold=10.0
)

print("=" * 50)
print("COMPLETENESS CHECK RESULTS")
print("=" * 50)

for check_name, result in completeness_results.items():
    status = "PASS" if result.passed else "FAIL"
    print(f"\n{check_name}: {status}")
    if result.errors:
        print(f"  Errors: {len(result.errors)}")
    if result.warnings:
        print(f"  Warnings: {len(result.warnings)}")

---
## Working with Pandas DataFrames

The DQA functions also work with pandas DataFrames (using DuckDB backend if Polars is unavailable).

In [ ]:
# Convert to pandas and run checks
labs_pandas = labs_df.to_pandas()

print("Testing with pandas DataFrame:")
print(f"DataFrame type: {type(labs_pandas)}")

# Run check with pandas DataFrame
result = check_required_columns(labs_pandas, labs_schema, 'labs')
print(f"\ncheck_required_columns - Passed: {result.passed}")

result = check_categorical_values(labs_pandas, labs_schema, 'labs')
print(f"check_categorical_values - Passed: {result.passed}")

---
## Working with LazyFrames

For large datasets, use Polars LazyFrames for memory-efficient processing.

In [ ]:
# Convert to LazyFrame
labs_lazy = labs_df.lazy()

print("Testing with Polars LazyFrame:")
print(f"DataFrame type: {type(labs_lazy)}")

# Run checks with LazyFrame (more memory efficient for large datasets)
result = check_required_columns(labs_lazy, labs_schema, 'labs')
print(f"\ncheck_required_columns - Passed: {result.passed}")

result = check_categorical_values(labs_lazy, labs_schema, 'labs')
print(f"check_categorical_values - Passed: {result.passed}")

result = check_missingness(labs_lazy, labs_schema, 'labs')
print(f"check_missingness - Passed: {result.passed}")

---
## Export Results to Dictionary

All result objects can be converted to dictionaries for serialization.

In [ ]:
# Export result to dictionary
result = check_required_columns(labs_df, labs_schema, 'labs')
result_dict = result.to_dict()

print("Result as dictionary:")
import json
print(json.dumps(result_dict, indent=2, default=str))

---
## Summary

### Conformance Checks:
- `check_table_exists()` - Verify table file exists
- `check_required_columns()` - Verify all required columns are present
- `check_column_dtypes()` - Verify column data types
- `check_datetime_format()` - Verify datetime format and timezone
- `check_lab_reference_units()` - Verify lab units match schema (labs table only)
- `check_categorical_values()` - Verify categorical values match mCIDE

### Completeness Checks:
- `check_missingness()` - Analyze missing values in required columns
- `check_conditional_requirements()` - Verify conditional required fields
- `check_mcide_value_coverage()` - Check coverage of mCIDE values
- `check_relational_integrity()` - Verify bidirectional foreign key coverage

### Orchestration:
- `run_conformance_checks()` - Run all conformance checks
- `run_completeness_checks()` - Run all completeness checks
- `run_relational_integrity_checks()` - Auto-detect and run FK checks across loaded tables
- `run_full_dqa()` - Run conformance + completeness + relational checks in one call